### Vectorized MicroGrad demo

This notebook mirrors `demo.ipynb`, but uses the NumPy-backed vectorized engine. The key idea is that one `Value` can hold an entire array, so a dense layer is built from a small number of array operations like `X @ W + b` instead of thousands of scalar `Value` operations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from vect_micrograd.vect_engine import Value
from vect_micrograd.vect_nn import MLP

np.random.seed(31337)


#### Make a checkerboard dataset

A checkerboard requires a more expressive network than the moon demo. This is exactly where scalar micrograd becomes slow, because every neuron decomposes into many tiny Python objects. The vectorized version keeps the same dynamic-autograd idea while letting NumPy do the dense array work.

In [ ]:
def make_checkerboard(n=400, grid=4, noise=0.02, seed=42):
    rng = np.random.default_rng(seed)
    side = int(np.sqrt(n))
    xs = np.linspace(-1, 1, side)
    ys = np.linspace(-1, 1, side)
    xx, yy = np.meshgrid(xs, ys)
    X = np.c_[xx.ravel(), yy.ravel()]

    ix = ((X[:, 0] + 1) / (2 / grid)).astype(int).clip(0, grid - 1)
    iy = ((X[:, 1] + 1) / (2 / grid)).astype(int).clip(0, grid - 1)
    y = ((ix + iy) % 2) * 2 - 1

    X += rng.normal(size=X.shape) * noise
    return X.astype(float), y.astype(float).reshape(-1, 1)

X, y = make_checkerboard(n=4000, grid=8, noise=0.02)

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), s=2, cmap='jet')
plt.title('Checkerboard dataset')
plt.show()

#### Initialize the vectorized model

`model.parameters()` now returns a small list of array-valued `Value` objects: one weight matrix and one bias vector per layer. The number of scalar trainable parameters is still large enough for the checkerboard, but the number of Python autograd nodes is much smaller.

In [ ]:
# Create a 4-layer network: 2 inputs → 16 hidden → ... → 16 hidden → 1 output
model = MLP(2, [128, 128, 128, 128, 1])

print(model)
print(f"number of scalar parameters: {sum(p.data.size for p in model.parameters()):,}")
print('number of Value parameter objects:', len(model.parameters()))

#### Optimization loop

In [ ]:
from vect_micrograd.optim import SGD, Adam, Lion
from vect_micrograd.utils import save_checkpoint, load_checkpoint, svm_loss, sample_batch

steps = 5000

# optimizer = SGD(model.parameters(), lr=1.0, total_steps=steps)
# optimizer = Adam(model.parameters(), lr=1e-2, total_steps=steps)
optimizer = Lion(model.parameters(), lr=1e-3, total_steps=steps, weight_decay=1e-2)

best_checkpoint = None
best_loss = float('inf')
best_step = -1
history = []

for k in range(steps):
    Xb, yb = sample_batch(X, y, batch_size=256)
    total_loss, acc = svm_loss(model, Xb, yb)

    loss_value = total_loss.item()
    history.append((k, loss_value, acc))

    if loss_value < best_loss:
        best_loss = loss_value
        best_step = k
        best_checkpoint = save_checkpoint(model)

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step(k)

    if k % 100 == 0:
        print(f'step {k:03d} loss {loss_value:.4f}, accuracy {acc*100:.1f}%')

#### Restore the best checkpoint and evaluate on the full dataset

In [ ]:
load_checkpoint(model, best_checkpoint)
full_loss, full_acc = svm_loss(model, X, y)

print(f'Restored checkpoint from step {best_step}, loss {best_loss:.4f}')
print(f'Full dataset: loss {full_loss.item():.4f}, accuracy {full_acc*100:.1f}%')


#### Training curve

In [ ]:
history = np.array(history)

plt.figure(figsize=(6, 4))
plt.plot(history[:, 0], history[:, 1], label='loss')
plt.plot(history[:, 0], history[:, 2], label='accuracy')
plt.xlabel('step')
plt.legend()
plt.title('Checkerboard test training')
plt.show()

#### Visualize the learned decision boundary

In [ ]:
h = 0.02
x_min, x_max = X[:, 0].min() - 0.25, X[:, 0].max() + 0.25
y_min, y_max = X[:, 1].min() - 0.25, X[:, 1].max() + 0.25
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Xmesh = np.c_[xx.ravel(), yy.ravel()]

scores = model(Value(Xmesh))
Z = (scores.data > 0).reshape(xx.shape)

plt.figure(figsize=(6, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), s=4, cmap=plt.cm.Spectral)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.title('Checkerboard test')
plt.show()

In [ ]:
from tests.test_value import (
    test_softmax_ce_probabilities_sum_to_one,
    test_softmax_ce_numerical_gradient,
    test_softmax_ce_loss_is_mean,
    test_softmax_ce_respects_upstream_grad,
)

test_softmax_ce_probabilities_sum_to_one()
test_softmax_ce_numerical_gradient()
test_softmax_ce_loss_is_mean()
test_softmax_ce_respects_upstream_grad()

print("All softmax_ce tests passed.")